# Anggota 4 — Ekstraksi Jawaban (Hybrid Linguistic-Symbolic)

Notebook ini mendemonstrasikan modul ekstraksi jawaban pada folder `Anggota4/` yang menggabungkan:
1. **Jalur Simbolik (`simbolik.py`)** untuk tabel interval (`NILAI_HURUF`, beban studi, bobot), rumus matematika, dan normalisasi bilangan kata.
2. **Jalur Linguistik (`ekstraksi.py`)** menggunakan dependency parsing `spacy-udpipe` (model `id`) dan skoring *lexical proximity*.
3. **Jalur Fallback** berbasis kecocokan kata kunci kalimat.

In [1]:
import json
import sys
from pathlib import Path

ROOT = Path.cwd()
if not (ROOT / "Anggota1_Korpus").exists() and (ROOT.parent / "Anggota1_Korpus").exists():
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from Anggota2_Pertanyaan.question_processing import proses_pertanyaan
from Anggota3_Retrieval.retrieval import Retrieval
from Anggota4.ekstraksi import EkstraktorJawaban
from Anggota4.simbolik import normalisasi_bilangan

korpus_dir = ROOT / "Anggota1_Korpus" / "data"
dokumen = json.loads((korpus_dir / "dokumen.json").read_text(encoding="utf-8"))
with (korpus_dir / "passages.jsonl").open(encoding="utf-8") as f:
    passages = [json.loads(line) for line in f if line.strip()]

retrieval = Retrieval(passages, dokumen)
ekstraktor = EkstraktorJawaban(dokumen)
print(f"Korpus dimuat: {len(passages)} passage")

Korpus dimuat: 440 passage


## 1. Normalisasi Bilangan Kata & Struktur Dependency Tree

In [2]:
contoh_teks = [
    "Waktu penyelesaian studi paling lama empat belas semester.",
    "Keterlambatan maksimal 15 (lima belas) menit.",
]
for teks in contoh_teks:
    print(teks, "->", normalisasi_bilangan(teks))

kalimat_jabatan = "Pengecualian Dosen Pembimbing dapat dilakukan atas usul Ketua Program Studi dan ditetapkan oleh Dekan."
doc = ekstraktor.nlp(kalimat_jabatan)
print("\nContoh token dependency (PROPN/VERB):")
for t in doc:
    if t.pos_ in {"PROPN", "VERB"}:
        print(f"  {t.text:14s} pos={t.pos_:5s} dep={t.dep_:10s} head={t.head.text}")

Waktu penyelesaian studi paling lama empat belas semester. -> Waktu penyelesaian studi paling lama 14 semester.
Keterlambatan maksimal 15 (lima belas) menit. -> Keterlambatan maksimal 15 menit.



Contoh token dependency (PROPN/VERB):


  Pengecualian   pos=PROPN dep=nsubj:pass head=dilakukan
  Dosen          pos=PROPN dep=flat       head=Pengecualian
  Pembimbing     pos=PROPN dep=flat       head=Dosen
  dilakukan      pos=VERB  dep=ROOT       head=dilakukan
  Program        pos=PROPN dep=flat       head=Ketua
  Studi          pos=PROPN dep=flat       head=Program
  ditetapkan     pos=VERB  dep=conj       head=Ketua
  Dekan          pos=PROPN dep=obl        head=ditetapkan


## 2. Uji End-to-End pada 4 Pertanyaan Rujukan Anggota 4 (`top_k=5`)

In [3]:
soal_a4 = json.loads((ROOT / "Anggota4" / "data" / "pertanyaan_anggota4.json").read_text(encoding="utf-8"))
for item in soal_a4:
    hq = proses_pertanyaan(item["pertanyaan"])
    kandidat = retrieval.cari_passage(hq, top_k=5)
    hasil = ekstraktor.ekstrak(hq, kandidat)
    print(f"Q: {item['pertanyaan']}")
    print(f"   Rujukan : {item['jawaban']}")
    print(f"   Ekstrak : {hasil['jawaban']} [{hasil['metode']}, {hasil['passage_id']}]")
    print(f"   Sumber  : {hasil['sumber']}\n")

Q: Berapa persentase kehadiran minimal agar mahasiswa boleh mengikuti UAS?
   Rujukan : 80%
   Ekstrak : 80% [linguistik_udpipe, pa2020-III-K-1]
   Sumber  : Buku Pedoman Akademik Fakultas Ilmu Komputer Universitas Brawijaya (2020), BAB III K, hlm. 30

Q: Berapa menit batas maksimal keterlambatan mengikuti ujian?
   Rujukan : 15 menit
   Ekstrak : 15 menit [linguistik_udpipe, tatatertib-X-S1-3]
   Sumber  : Tata Tertib Pelaksanaan Ujian (UTS & UAS) FILKOM UB (2016), PENGUMUMAN TATA TERTIB MAHASISWA PESERTA UJIAN, hlm. 1

Q: Berapa bobot nilai huruf B+?
   Rujukan : 3,5
   Ekstrak : 3,5 [simbolik, pa2020-bobot-2]
   Sumber  : Buku Pedoman Akademik Fakultas Ilmu Komputer Universitas Brawijaya (2020), BAB II H, hlm. 19



Q: Siapa yang menetapkan pengecualian Dosen Pembimbing Skripsi?
   Rujukan : Dekan
   Ekstrak : Dekan [linguistik_udpipe, pa2020-VI-A-11]
   Sumber  : Buku Pedoman Akademik Fakultas Ilmu Komputer Universitas Brawijaya (2020), BAB VI A, hlm. 41



## 3. Uji Isolasi Seluruh 11 Pertanyaan Rujukan (Anggota 2, 3, 4)

In [4]:
p_by_id = {p["id"]: p for p in passages}
berkas_uji = (
    ROOT / "Anggota2_Pertanyaan" / "data" / "pertanyaan_anggota2.json",
    ROOT / "Anggota3_Retrieval" / "data" / "pertanyaan_anggota3.json",
    ROOT / "Anggota4" / "data" / "pertanyaan_anggota4.json",
)
for path in berkas_uji:
    for item in json.loads(path.read_text(encoding="utf-8")):
        hq = proses_pertanyaan(item["pertanyaan"])
        kandidat = [
            {**p_by_id[pid], "passage_id": pid, "rank": i, "score": 1.0, "sumber": dict(dokumen[p_by_id[pid]["dok_id"]])}
            for i, pid in enumerate(item["passage_id"], start=1)
        ]
        hasil = ekstraktor.ekstrak(hq, kandidat)
        print(f"[{item['penanya']}] {item['pertanyaan']} -> {hasil['jawaban']} ({hasil['metode']})")

[Anggota 2] Berapa sks minimal untuk menyelesaikan studi sarjana? -> 144 sks (linguistik_udpipe)
[Anggota 2] Berapa lama waktu maksimal untuk Ujian Skripsi? -> 75 menit (linguistik_udpipe)
[Anggota 2] Apa itu skripsi? -> Karya ilmiah mahasiswa pendidikan program sarjana yang merupakan wujud dari kajian pengetahuan dan/atau penerapan teknologi berdasarkan kaidah ilmiah dalam bidang studi yang telah dipelajarinya. (linguistik_udpipe)
[Anggota 2] Apa nilai huruf untuk nilai angka 78? -> B+ (simbolik)


[Anggota 3] Berapa lama waktu maksimal untuk menyelesaikan studi sarjana? -> 14 semester (linguistik_udpipe)
[Anggota 3] Kapan batas pengajuan cuti akademik setelah daftar ulang? -> Paling lambat 1 bulan setelah berakhirnya masa daftar ulang. (linguistik_udpipe)
[Anggota 3] Apa rumus indeks prestasi semester? -> IP = Σ (K_i × Na_i) / Σ K_i. (simbolik)
[Anggota 4] Berapa persentase kehadiran minimal agar mahasiswa boleh mengikuti UAS? -> 80% (linguistik_udpipe)
[Anggota 4] Berapa menit batas maksimal keterlambatan mengikuti ujian? -> 15 menit (linguistik_udpipe)
[Anggota 4] Berapa bobot nilai huruf B+? -> 3,5 (simbolik)
[Anggota 4] Siapa yang menetapkan pengecualian Dosen Pembimbing Skripsi? -> Dekan (linguistik_udpipe)
